# Welcome to NER Finetuning with HF Transformers 

## Contents 
1. Configurations (Resources, Training, Optimization, Evaluation, Saving results)
2. Prepare training & dataset
3. Train (Finetune) a NER Model
4. Optimization
5. Evaluation

In [1]:
# import modules
import config
import train
import eval_opt
import merge_datasets

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train.set_torch_device()

device(type='cuda')

## 1. Configurations

In this section, we set all of the configurations for the following steps (training, optimization, evaluation). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

### Load Model

You can choose e.g. one of the following models and add them as `path` in the model config:
* FacebookAI/roberta-base
* dbmdz/bert-base-german-cased
* dbmdz/bert-base-historic-multilingual-cased
* flair/ner-german (not implemented so far)
* distilbert/distilbert-base-uncased

In [3]:
model = config.Resource(path="dbmdz/bert-base-german-europeana-uncased", source="hf")
model.set_name()
print(model.info())

bert-base-german-europeana-uncased will be loaded from dbmdz/bert-base-german-europeana-uncased (via hf).


### Load and merge dataset(s)

You can choose e.g. one of the following dataset configurations:
* `path="data/zefys2025.hf", source="local"`
* `path="eriktks/conll2003", source="hf"`
* `path="GermanEval/germeval_14", source="hf"`
* `path="data/hisgermaner.hf", source="local"`
* `path="data/newseye.hf", source="local"`
* `path="data/hipe2020.hf", source="local"`

In [4]:
zefys_ds = config.Resource(path="data/zefys2025_not-casted.hf", source="local")
zefys_ds.set_name()
print(zefys_ds.info())

zefys2025_not-casted will be loaded from data/zefys2025_not-casted.hf (via local).


In [5]:
hipe_ds = config.Resource(path="data/hipe2020_not-casted.hf", source="local")
hipe_ds.set_name()
print(hipe_ds.info())

hipe2020_not-casted will be loaded from data/hipe2020_not-casted.hf (via local).


In [6]:
hisgermaner_ds = config.Resource(path="data/hisgermaner_not-casted.hf", source="local")
hisgermaner_ds.set_name()
print(hisgermaner_ds.info())

hisgermaner_not-casted will be loaded from data/hisgermaner_not-casted.hf (via local).


In [7]:
zefys_dataset = train.load_ner_dataset(zefys_ds.path, zefys_ds.source)
hipe_dataset = train.load_ner_dataset(hipe_ds.path, hipe_ds.source)
hisgermaner_dataset = train.load_ner_dataset(hisgermaner_ds.path, hisgermaner_ds.source)

In [8]:
merged_dataset = merge_datasets.merge_ds([zefys_dataset, hipe_dataset, hisgermaner_dataset])

In [9]:
from datasets import Sequence, ClassLabel
merged_label_list = merge_datasets.get_label_list(merged_dataset["train"]["ner_tags"])
merged_dataset = merged_dataset.cast_column("ner_tags", Sequence(ClassLabel(names=merged_label_list)))

In [10]:
merged_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 18234
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3262
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3258
    })
})

In [19]:
merged_ds = config.Resource(path="hipe-zefys-hisgermaner", source="local")
merged_ds.set_name()

## 2. Prepare dataset
choose one of [2a, 2b] - 2a for simple tokenization (e.g. single dataset), 2b for tokenization based on updated labels

### 2a. Simple Tokenization
example: hipe dataset

In [ ]:
"""
# for roberta
import torch
from datetime import datetime
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import eval_opt
import evaluate
import numpy as np

def get_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, add_prefix_space=True)
    return tokenizer

tokenizer = get_tokenizer(model.path)
"""

In [ ]:
tokenizer = train.get_tokenizer(model.path)

In [ ]:
hipe_dataset["train"].features["ner_tags"]

In [ ]:
hipe_label_list = merge_datasets.get_label_list(hipe_dataset["train"]["ner_tags"])
hipe_dataset = hipe_dataset.cast_column("ner_tags", Sequence(ClassLabel(names=hipe_label_list)))
hipe_tokenized_dataset = train.prepare_dataset(hipe_dataset, tokenizer)

### 2b. Tokenization on updated labels
example: merged dataset
all labels (`ner_tags`) that do not occur in our zefys2025 dataset will be removed before tokenization

In [11]:
#ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
label_list = train.get_label_list(merged_dataset)
merged_dataset["train"].features["ner_tags"]

Sequence(feature=ClassLabel(names=['B-LOC', 'B-ORG', 'B-PER', 'B-PROD', 'B-TIME', 'I-LOC', 'I-ORG', 'I-PER', 'I-PROD', 'I-TIME', 'O'], id=None), length=-1, id=None)

In [12]:
dataset_updated = merge_datasets.drop_ner_labels(label_list, merged_dataset)

Casting the dataset: 100%|█████████████████████████████████████████████████| 3258/3258 [00:00<00:00, 8794.45 examples/s]


In [13]:
#as we can see now, the labels from tokenized dataset that are not in our zefys_label_list are removed from the ClassLabels (replaced with "O")
print(dataset_updated["train"].features["ner_tags"])

Sequence(feature=ClassLabel(names=['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O'], id=None), length=-1, id=None)


In [14]:
#also, the corresponding ids (nums) for our ner_tags have been updated
print(merged_dataset["train"][2])
print(dataset_updated["train"][2])

{'id': 5741, 'tokens': ['Landwirtschaftsminister', 'Georges', 'Monnet', 'wurde', '1898', 'geboren', 'und', 'ist', 'seit', '1928', 'Abgeordneter', '.'], 'ner_tags': [10, 2, 7, 10, 10, 10, 10, 10, 10, 10, 10, 10]}
{'id': 5741, 'tokens': ['Landwirtschaftsminister', 'Georges', 'Monnet', 'wurde', '1898', 'geboren', 'und', 'ist', 'seit', '1928', 'Abgeordneter', '.'], 'ner_tags': [6, 2, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6]}


In [15]:
tokenizer = train.get_tokenizer(model.path)
tokenized_dataset = train.prepare_dataset(dataset_updated, tokenizer)
label_list = train.get_label_list(dataset_updated)

Map: 100%|████████████████████████████████████████████████████████████████| 3262/3262 [00:00<00:00, 13076.83 examples/s]


## 3. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

### Training Parameters

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [16]:
training_params = config.TrainingParams()
training_params.__dict__

{'batch_size': 16,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01,
 'warmup_steps': 100}

In [17]:
training_params.batch_size = 32
training_params.num_train_epochs = 1
training_params.__dict__

{'batch_size': 32,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 1,
 'weight_decay': 0.01,
 'warmup_steps': 100}

In [20]:
trained_ner_model, model_out_path = train.train_model(model, merged_ds, label_list, training_params, tokenized_dataset, tokenizer)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dbmdz/bert-base-german-europeana-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.194400,0.133303,0.605596,0.629858,0.617489,0.957429


## 4. Optimization

### Evaluation and optimization

* Evaluation: Stratified k-fold Cross-Validation - one of ["None", k] (https://huggingface.co/docs/datasets/v1.11.0/splits.html#examples; https://discuss.huggingface.co/t/k-fold-cross-validation/5765/5, https://github.com/huggingface/datasets/discussions/5940)
* Optimization:
  * https://huggingface.co/docs/transformers/hpo_train
  * https://medium.com/carbon-consulting/transformer-models-hyperparameter-optimization-with-the-optuna-299e185044a8
  * Grid Search (GridSearchCV?), Line Search, hyperopt
* Analyze grid search results using Heatmap or similar

### Optimization

For optimization (hyperparameter search) we use the HF Trainer API (https://huggingface.co/docs/transformers/hpo_train#hyperparameter-search-using-trainer-api) and optuna (https://optuna.readthedocs.io/en/stable/index.html, `pip install optuna`). We can access all of the search parameters, as they are stored in a dict structure, and also adapt the number of trials `n_trials` to run. 

In [ ]:
optimize_params = config.OptimizeParams()

optimize_params.hp_space["learning_rate"]["param_type"]
optimize_params.n_trials = 10
optimize_params.__dict__

In [ ]:
#to remove an hyperparameter from the hp_space dict, use pop() or similar
optimize_params.hp_space.pop("warmup_steps")
optimize_params.__dict__

In [ ]:
model_out_path = "roberta_germeval14_opt/"
eval_opt.optimize(optimize_params, training_params, model.path, model_out_path, label_list, tokenizer, tokenized_dataset)

## 5. Evaluation

In [21]:
class_report, errors = eval_opt.compute_metrics_per_tag(trained_ner_model, tokenized_dataset, label_list) 

              precision    recall  f1-score   support

         LOC       0.68      0.78      0.73      2502
         ORG       0.37      0.37      0.37       776
         PER       0.54      0.56      0.55      1756

   micro avg       0.59      0.64      0.61      5034
   macro avg       0.53      0.57      0.55      5034
weighted avg       0.58      0.64      0.61      5034



In [ ]:
eval_opt.save_class_report(class_report, "md", model_out_path)

In [ ]:
config.save_train_config(model_out_path, training_params)

In [ ]:
#inference example
from transformers import pipeline, AutoModelForTokenClassification

text = "Novelle	von Joh. L. Buchta."

id2label = {0:label_list[0], 1:label_list[1], 2:label_list[2], 3:label_list[3], 4:label_list[4], 5:label_list[5], 6:label_list[6]}
finetuned_model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base_hisgermaner_2025-04-02_14-45/checkpoint-276", num_labels=7, id2label=id2label)
classifier = pipeline("ner", model=finetuned_model, tokenizer=tokenizer)
classifier(text)